In [ ]:
%pip install pandas
%pip install xgboost
%pip install sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score


In [ ]:
customers_data = pd.read_csv('data_sets/customers.csv')

# CHECK FOR NULL VALUES
customers_data.isnull().sum()

customers_data

In [ ]:
merchants_data = pd.read_csv('data_sets/merchants.csv')

# CHECK FOR NULL VALUES
merchants_data.isnull().sum()

In [ ]:
# DROP COLUMNS THAT ARE NOT NEEDED
merchants_data = merchants_data.drop(columns=['BUSINESS_TYPE', 'MCC_CODE', 'LEGAL_NAME', 'FOUNDATION_DATE',
                                                'TAX_EXCEMPT_INDICATOR', 'ACTIVE_FROM', 'TRADING_FROM', 'ANNUAL_TURNOVER_CARD',
                                                  'AVERAGE_TICKET_SALE_AMOUNT', 'DELIVERY_SAME_DAYS_PERCENTAGE',
                                                    'DELIVERY_WEEK_ONE_PERCENTAGE', 'DELIVERY_WEEK_TWO_PERCENTAGE', 
                                                    'DELIVERY_OVER_TWO_WEEKS_PERCENTAGE',
                                                    'PAYMENT_PERCENTAGE_FACE_TO_FACE','PAYMENT_PERCENTAGE_ECOM',
                                                    'PAYMENT_PERCENTAGE_MOTO'])

# CALCULATE DIFFERENCE IN DEPOSIT_REQUIRED_PERCENTAGE AND DEPOSIT_PERCENTAGE AND SAVE TO DEPOSIT_DIFF
merchants_data['DEPOSIT_DIFF'] = merchants_data['DEPOSIT_REQUIRED_PERCENTAGE'] - merchants_data['DEPOSIT_PERCENTAGE']

# DROP THE INDIVIDUAL DEPOSIT_PERCENTAGE AND DEPOSIT_REQUIRED_PERCENTAGE COLUMNS
merchants_data = merchants_data.drop(columns=['DEPOSIT_REQUIRED_PERCENTAGE', 'DEPOSIT_PERCENTAGE'])

merchants_data

In [ ]:
terminals_data = pd.read_csv('data_sets/terminals.csv')

# CHECK FOR NULL VALUES
terminals_data.isnull().sum()

terminals_data

In [ ]:
transactions_test_data = pd.read_csv('data_sets/transactions_test.csv')

transactions_test_data = pd.merge(transactions_test_data, customers_data, on='CUSTOMER_ID')
transactions_test_data = pd.merge(transactions_test_data, terminals_data, on='TERMINAL_ID')

# # EUCLIDIAN DISTANCE
transactions_test_data['distance'] = (((transactions_test_data['x_customer_id'] - transactions_test_data['y_customer_id'])**2)
                                + ((transactions_test_data['x_terminal_id'] - transactions_test_data['y_terminal__id'])**2))**0.5
# CHECK FOR NULL VALUES
transactions_test_data.isnull().sum()

In [ ]:
# DROP COLUMNS
transactions_test_data = transactions_test_data.drop(columns=['TX_TS', 'TX_AMOUNT', 'FAILURE_CODE',
                                                                'FAILURE_REASON', 'CARD_BRAND', 'CARD_DATA', 'CARD_EXPIRY_DATE',
                                                                  'TRANSACTION_STATUS', 'TRANSACTION_TYPE',
                                                                  'CARD_COUNTRY_CODE','TRANSACTION_CURRENCY',
                                                                  'TRANSACTION_GOODS_AND_SERVICES_AMOUNT',
                                                                  'x_customer_id', 'y_customer_id',
                                                                  'x_terminal_id', 'y_terminal__id'])

# DROP COLUMN IF ALL VALUES ARE 0
transactions_test_data = transactions_test_data.loc[:, (transactions_test_data != 0.0).any(axis=0)]

transactions_test_data

In [ ]:
# CHECK FOR NULL VALUES AGAIN
transactions_test_data.isnull().sum()

In [ ]:
transactions_train_data = pd.read_csv('data_sets/transactions_train.csv')

transactions_train_data = pd.merge(transactions_train_data, customers_data, on='CUSTOMER_ID')
transactions_train_data = pd.merge(transactions_train_data, terminals_data, on='TERMINAL_ID')

# EUCLIDIAN DISTANCE
transactions_train_data['distance'] = (((transactions_train_data['x_customer_id'] - transactions_train_data['y_customer_id'])**2)
                                + ((transactions_train_data['x_terminal_id'] - transactions_train_data['y_terminal__id'])**2))**0.5


# CHECK FOR NULL VALUES
transactions_train_data.isnull().sum()

In [ ]:
# Clean data:
transactions_train_data = transactions_train_data.drop(columns=['TX_ID', 'TX_TS', 'TX_AMOUNT', 'FAILURE_CODE',
                                                                 'FAILURE_REASON', 'CARD_BRAND', 'CARD_DATA',
                                                                   'CARD_EXPIRY_DATE', 'TRANSACTION_STATUS', 'TRANSACTION_TYPE',
                                                                   'CARD_COUNTRY_CODE','TRANSACTION_CURRENCY',
                                                                  'TRANSACTION_GOODS_AND_SERVICES_AMOUNT',
                                                                  'x_customer_id', 'y_customer_id',
                                                                  'x_terminal_id', 'y_terminal__id'])

# DROP COLUMN IF ALL VALUES ARE 0
transactions_train_data = transactions_train_data.loc[:, (transactions_train_data != 0.0).any(axis=0)]

# DROP ROW IF A VALUE IS NAN
transactions_train_data = transactions_train_data.dropna()

transactions_train_data

In [ ]:
# CHECK FOR NULL VALUES AGAIN
transactions_train_data.isnull().sum()

In [ ]:
# MERGE TABLES FOR TRAINING DATA
train_data = pd.merge(transactions_train_data, merchants_data, on='MERCHANT_ID', how='left')

train_data

In [ ]:
# PRINT ANY ENTRIES THAT ARE NA
print(train_data[train_data.isna().any(axis=1)])

In [ ]:
# MERGE TABLES FOR TESTING DATA
test_data = pd.merge(transactions_test_data, merchants_data, on='MERCHANT_ID', how='left')

test_data

In [ ]:
# PRINT ANY ENTRIES THAT ARE NA
test_data[test_data.isna().any(axis=1)]

In [ ]:
# Separate target prediction
y = train_data.TX_FRAUD
X = train_data.drop(['TX_FRAUD' ,'CUSTOMER_ID',  'TERMINAL_ID','MERCHANT_ID'], axis=1)
test_data_TX_ID = test_data.TX_ID
X_test = test_data.drop(['CUSTOMER_ID',  'TERMINAL_ID','MERCHANT_ID', 'ID_JOIN', 'TX_ID'], axis=1)
test_data_TX_ID.head()

In [ ]:
# Divide data into training data and validation data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2,random_state=0)

object_cols = [col for col in X_train.columns if X_train[col].dtype == "object"]
object_cols_test_data = [col for col in X_train.columns if X_train[col].dtype == "object"]

In [ ]:
# Apply one-hot encoder to each column with categorical data
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[object_cols].astype(str)))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(X_valid[object_cols].astype(str)))
OH_cols_test = pd.DataFrame(OH_encoder.transform(X_test[object_cols].astype(str)))

In [ ]:
# One-hot encoding removed index; put it back
OH_cols_train.index = X_train.index
OH_cols_valid.index = X_valid.index
OH_cols_test.index = X_test.index

In [ ]:
# Remove categorical columns (will replace with one-hot encoding)
num_X_train = X_train.drop(object_cols, axis=1)
num_X_valid = X_valid.drop(object_cols, axis=1)
num_X_test = X_test.drop(object_cols_test_data, axis=1)

In [ ]:
# Add one-hot encoded columns to numerical features
OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)
OH_X_test = pd.concat([num_X_test, OH_cols_test], axis=1)

In [ ]:
OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)
OH_X_test.columns = OH_X_test.columns.astype(str)

In [ ]:
OH_X_train.head()

In [ ]:
reg_model = XGBRegressor(eval_metric=mean_absolute_error, learning_rate=0.05, n_estimators=500)
reg_model.fit(OH_X_train, y_train, eval_set=[(OH_X_valid, y_valid)], early_stopping_rounds=5, verbose=False)

In [ ]:
reg_model.evals_result()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

def round_data(value):
    if value < 0.5:
        return 0
    else:
        return 1
    
validation_data_predictions = reg_model.predict(OH_X_valid)
data = {'Predicted': validation_data_predictions,'Expected': y_valid}
df = pd.DataFrame(data)
y_true = df['Expected']

validation_data_predictions = df['Predicted'].apply(round_data)

accuracy = accuracy_score(y_valid, validation_data_predictions)
precision = precision_score(y_true, validation_data_predictions)
recall = recall_score(y_true, validation_data_predictions)
auc = roc_auc_score(y_true, validation_data_predictions)
 
print("Accuracy: {:.2f}".format(accuracy * 100))
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print("AUC:", auc)

In [ ]:
answers = reg_model.predict(OH_X_test)
answers

In [ ]:
output = pd.DataFrame({"TX_ID": test_data_TX_ID, "TX_FRAUD": answers})
output.set_index("TX_ID", inplace=True)
output

In [ ]:
max_a = output.TX_FRAUD.max()
min_a = output.TX_FRAUD.min()
min_norm = 0
max_norm =1
output['TX_FRAUD'] = round((output.TX_FRAUD- min_a) * (max_norm - min_norm) / (max_a-min_a) + min_norm , 0)
output

In [ ]:
output.to_csv('submission.csv')